In [33]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [34]:
load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [35]:
class BatsmanState(TypedDict):

    runs: int
    balls: int
    fours: int
    sixes: int

# calculate strike rate, balls per boundary etc
    sr: float
    bpb: float
    boundary_percent: float
    summary: str


In [36]:
def cal_sr(state: BatsmanState):

    sr = (state['runs']/ state['balls']) * 100

    return {'sr': sr}


In [37]:
def cal_bpb(state: BatsmanState):

    bpb = state['balls']/ state['fours'] + state['sixes']

    return {'bpb': bpb}

In [38]:
def cal_bp(state: BatsmanState):

    bp = (((state['fours'] * 4) + (state['sixes'] * 6))/ state['runs']) * 100

    return {'boundary_percent': bp}

In [39]:
def summary(state: BatsmanState):

    summary = f"""
    Strike Rate - {state['sr']} \n
    Balls per boundary - {state['bpb']} \n
    Boundary Percent - {state['boundary_percent']}
    """

    return {'summary': summary}

In [40]:
graph = StateGraph(BatsmanState)

# nodes
graph.add_node('cal_sr', cal_sr)
graph.add_node('cal_bpb', cal_bpb)
graph.add_node('cal_bp', cal_bp)
graph.add_node('summary', summary)

# edges
graph.add_edge(START, 'cal_sr')
graph.add_edge(START, 'cal_bpb')
graph.add_edge(START, 'cal_bp')

graph.add_edge('cal_sr', 'summary')
graph.add_edge('cal_bpb', 'summary')        
graph.add_edge('cal_bp', 'summary') 

graph.add_edge('summary', END)

workflow = graph.compile()

      

In [41]:
initial_state = {
    'runs': 100, 'balls': 60, 'fours': 10, 'sixes': 5
    }

final_state = workflow.invoke(initial_state)
print(final_state)

{'runs': 100, 'balls': 60, 'fours': 10, 'sixes': 5, 'sr': 166.66666666666669, 'bpb': 11.0, 'boundary_percent': 70.0, 'summary': '\n    Strike Rate - 166.66666666666669 \n\n    Balls per boundary - 11.0 \n\n    Boundary Percent - 70.0\n    '}


In [42]:
print(final_state['summary'])


    Strike Rate - 166.66666666666669 

    Balls per boundary - 11.0 

    Boundary Percent - 70.0
    
